<a href="https://colab.research.google.com/github/princesspretty-b/masai_assignments/blob/main/ml-leakage-pipeline-bhuvaneshwari/Avoiding_Traps_Leakage.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Task 1 — Reproduce and Identify Leakage:

    Using the synthetic dataset generated below, scale the features on the entire dataset before splitting, train a Logistic Regression model, and report train and test accuracy.
    Identify what is wrong with this approach.

In [1]:
from sklearn.datasets import make_classification
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Generate data
X, y = make_classification(n_samples=1000, n_features=10, random_state=42)

# WRONG: scaling before split
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# Train model
model = LogisticRegression()
model.fit(X_train, y_train)

# Evaluate
train_acc = accuracy_score(y_train, model.predict(X_train))
test_acc = accuracy_score(y_test, model.predict(X_test))

print("Train Accuracy:", train_acc)
print("Test Accuracy:", test_acc)

Train Accuracy: 0.87
Test Accuracy: 0.83


Explanation:

The scaler fit function is applied on the entire dataset and not only on the trainig data.
Hence the model learns mean and variance from the entire dataset, including the test set.
This introduces data leakage, because test data influences training transformations and the Model performance becomes over-optimistic

Task 2 — Fix the Workflow Using a Pipeline:

    a. Refactor the code from Task 1 using a Pipeline that combines StandardScaler and LogisticRegression.
    b. Split the data first using train_test_split, then run 5-fold cross-validation using cross_val_score.
    c. Report mean accuracy and standard deviation.

In [2]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Generate data
X, y = make_classification(n_samples=1000, n_features=10, random_state=42)

# Split first
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression())
])

# Cross-validation
scores = cross_val_score(pipeline, X_train, y_train, cv=5)

print("CV Mean Accuracy:", scores.mean())
print("CV Std Dev:", scores.std())

CV Mean Accuracy: 0.8699999999999999
CV Std Dev: 0.01785357107135714


Explanation:
*   Scaling happens inside each fold, using only training data of that fold
*   No leakage from validation folds
*   Cross-validation gives a more reliable estimate of performance
*   Std.dev value is very less comaped to the mean accuracy shows the model work well

Task 3 — Experiment with Decision Tree Depth:

    a. Train a DecisionTreeClassifier at max_depth values of 1, 5, and 20 using the same train-test split from Task 2.
    b. Record train and test accuracy for each depth in a table  
    c. Briefly explain which depth best balances fit and generalization.

In [6]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

depths = [1, 5, 20]
results = []

for d in depths:
    model = DecisionTreeClassifier(max_depth=d, random_state=42)
    model.fit(X_train, y_train)

    train_acc = accuracy_score(y_train, model.predict(X_train))
    test_acc = accuracy_score(y_test, model.predict(X_test))

    results.append((d, train_acc, test_acc))

# Print results
print("Depth | Train Accuracy | Test Accuracy")
for r in results:
    print(f"{r[0]:5} | {r[1]:14.3f} | {r[2]:13.3f}")

Depth | Train Accuracy | Test Accuracy
    1 |          0.882 |         0.840
    5 |          0.954 |         0.855
   20 |          1.000 |         0.840


Explanation:

*   Depth = 5 is the best choice as it demonstrates High test accuracy and Smaller gap between train/test
*   Depth 20 clearly overfits (perfect train accuracy but worse test performance)